I downlod data of cps_bls from :
https://www.bls.gov/cps/tables.htm
I use the data of 2019 and 2020 to compare the employment rate of different groups of people.


In [ ]:
using DataFrames, CSV, Dates, Plots, StatsPlots, ExcelFiles, XLSX, ExcelReaders

In [ ]:
using XLSX
file = XLSX.readxlsx("D:\\PSID_SHELF\\02_NLSY79\\Karlanser\\CPS\\CPS-BLS\\compare_psid\\PANEL_CPS.xlsx")
sheets = XLSX.sheetnames(file)
println(sheets)


In [ ]:
# Import data from Excel file
df = DataFrame(XLSX.readtable(file_path, "Sheet2")...)

In [ ]:
# Rename columns
rename!(df, [
    :LNU02000000 => :Employment_Level,
    :LNU07000000 => :EE,
    :LNU07100000 => :UE,
    :LNU07200000 => :IE,
    :LNU07300000 => :ME,
    :LNU03000000 => :Unemployment_Level,
    :LNU07400000 => :EU,
    :LNU07500000 => :UU,
    :LNU07600000 => :IU,
    :LNU07700000 => :MU,
    :LNU05000000 => :Not_in_Labor_Force,
    :LNU07800000 => :EI,
    :LNU07900000 => :UI,
    :LNU08000000 => :II,
    :LNU08100000 => :MI,
    :LNU08200000 => :EM,
    :LNU08300000 => :UM,
    :LNU08400000 => :IM
])


In [ ]:
# Define a dictionary to map month abbreviations to numbers
month_dict = Dict("Jan" => 1, "Feb" => 2, "Mar" => 3, "Apr" => 4, "May" => 5, "Jun" => 6, 
                  "Jul" => 7, "Aug" => 8, "Sep" => 9, "Oct" => 10, "Nov" => 11, "Dec" => 12)

# Extract month and year from YEAR_MONTH
df.month = map(x -> month_dict[x[1:3]], df.YEAR_MONTH)
df.year = map(x -> parse(Int, x[5:end]), df.YEAR_MONTH)

In [ ]:
df

In [ ]:
#sort by year and month
df = sort(df, [:year, :month])

In [ ]:
# Generate lagged variables for each labor market status
df.Employment_Level_lag = [missing; df.Employment_Level[1:end-1]]
df.Unemployment_Level_lag = [missing; df.Unemployment_Level[1:end-1]]
df.Not_in_Labor_Force_lag = [missing; df.Not_in_Labor_Force[1:end-1]]


In [ ]:
# Calculate transition variables using log transformation
df.tr_EE = -log.(1 .- (df.EE ./ df.Employment_Level_lag))
df.tr_UU = -log.(1 .- (df.UU ./ df.Unemployment_Level_lag))
df.tr_II = -log.(1 .- (df.II ./ df.Not_in_Labor_Force_lag))
df.tr_EU = -log.(1 .- (df.EU ./ df.Employment_Level_lag))
df.tr_UE = -log.(1 .- (df.UE ./ df.Unemployment_Level_lag))
df.tr_IE = -log.(1 .- (df.IE ./ df.Employment_Level_lag))
df.tr_EI = -log.(1 .- (df.EI ./ df.Employment_Level_lag))
df.tr_IU = -log.(1 .- (df.IU ./ df.Not_in_Labor_Force_lag))
df.tr_UI = -log.(1 .- (df.UI ./ df.Unemployment_Level_lag))


In [ ]:
#tsset to set panel data for grapgh transition
df.ts = df.year .+ (df.month .- 1) ./ 12

In [ ]:
# Define a function to plot and save figures
function plot_and_save(df, x, y, title, filename)
    p = plot(df[!, x], df[!, y], title=title,
             xlabel="Month", ylabel="Transition Rate", legend=false)
    if isfile(filename)
        rm(filename)  # remove the file if it already exists
    end
    savefig(p, filename)
end

# Call the function for each transition rate
plot_and_save(df, :ts, :tr_EU, "Transition Rate: Employment to Unemployment", "tr_EU.png")
plot_and_save(df, :ts, :tr_UE, "Transition Rate: Unemployment to Employment", "tr_UE.png")
plot_and_save(df, :ts, :tr_EI, "Transition Rate: Employment to Inactivity", "tr_EI.png")
plot_and_save(df, :ts, :tr_IE, "Transition Rate: Inactivity to Employment", "tr_IE.png")
plot_and_save(df, :ts, :tr_UI, "Transition Rate: Unemployment to Inactivity", "tr_UI.png")
plot_and_save(df, :ts, :tr_IU, "Transition Rate: Inactivity to Unemployment", "tr_IU.png")
plot_and_save(df, :ts, :tr_EE, "Transition Rate: Employment to Employment", "tr_EE.png")
plot_and_save(df, :ts, :tr_UU, "Transition Rate: Unemployment to Unemployment", "tr_UU.png")
plot_and_save(df, :ts, :tr_II, "Transition Rate: Inactivity to Inactivity", "tr_II.png")


In [ ]:
# Load the additional dataset
nlsy79_df = CSV.read("nlsy79_data.csv", DataFrame)


In [ ]:

# Merge the two datasets on the 'date_num' column
merged_df = innerjoin(df, nlsy79_df, on = :date_num)
